# Cleaning the Animals Dataset (pandas)
Run the cells **from top to bottom**. Each step fixes one type of problem.

## Step 1 - Upload and load the file
Run this cell, click **Choose files**, and select `animals_raw.csv`.

In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [4]:
import pandas as pd
import numpy as np
import re

# dtype=str means "read everything as text", so pandas does not guess wrong
df = pd.read_csv("animals_raw.csv", dtype=str)
df.head()

,animal_id,common_name,scientific_name,class,diet,habitat,country,facility,conservation_status,sex,age_years,weight_kg,body_length_cm,lifespan_years,daily_food_kg,health_status,vaccinated,checkup_cost_usd,date_recorded
0,ZA-04453,Tiger,Panthera tigris,Mammal,Carnivore,Forest,India,Metro City Zoo,Endangered,Male,10.9,297.52,291.5,-1,14.24,Healthy,1,216.29,2025-06-21
1,ZA-00434,Giant Panda,Ailuropoda melanoleuca,Mammal,Herbivore,Forest,China,Greenfield Sanctuary,Vulnerable,male,10.1,105.82,165.4,20,NaN,Healthy,Yes,129.65,2025-08-11
2,ZA-02153,Lion,Panthera leo,Mammal,Carnivore,Savanna,Tanzania,Greenfield Sanctuary,Vulnerable,Female,5.8,180.22,230.3,14,5.72,Healthy,No,211.39,16/05/2022
3,ZA-04341,Plains Zebra,Equus quagga,Mammal,Herbivore,Savanna,Tanzania,Coastal Aquarium & Zoo,Near Threatened,Female,NaN,294.8 lbs,176.0,25,7.83,Healthy,Yes,$137.20,03/09/2023
4,ZA-02905,Chimpanzee,Pan troglodytes,Mammal,Omnivore,Forest,Uganda,Metro City Zoo,-,Female,20.9,NaN,86.1,40,2.22,Healthy,Yes,87.31,2023-01-21


## Step 2 - Look at the data first


In [5]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
# print()
print(df.isna().sum())

Shape: (5150, 19)
Duplicate rows: 148
animal_id                0
common_name              0
scientific_name          0
class                    0
diet                     0
habitat                 60
country                 63
facility                59
conservation_status     54
sex                     89
age_years              302
weight_kg              168
body_length_cm         140
lifespan_years           0
daily_food_kg          375
health_status           59
vaccinated             191
checkup_cost_usd       243
date_recorded           66
dtype: int64


## Step 3 - Remove extra spaces and make missing values real `NaN`
The file uses many words for 'missing': `N/A`, `-`, `unknown`, `NULL` ...

In [8]:
# remove spaces at the start/end of every text value
for col in df.columns:
    df[col] = df[col].str.strip()

# turn all "missing" words into real NaN
missing_words = ["", "N/A", "NA", "na", "NULL", "null", "-", "unknown", "Unknown", "None"]
df = df.replace(missing_words, np.nan)

df.isna().sum()

,0
animal_id,0
common_name,0
scientific_name,0
class,0
diet,0
habitat,104
country,103
facility,101
conservation_status,105
sex,153


## Step 4 - Fix IDs and remove duplicates

In [9]:
df["animal_id"] = df["animal_id"].str.upper()

# check the format: ZA- followed by 5 digits
valid_id = df["animal_id"].str.match(r"^ZA-\d{5}$", na=False)
print("Bad IDs:", (~valid_id).sum())
df = df[valid_id]

# keep one row per animal (the row with the fewest missing values)
df["filled"] = df.notna().sum(axis=1)
df = df.sort_values("filled", ascending=False).drop_duplicates("animal_id")
df = df.drop(columns="filled").sort_index()

print("Rows left:", len(df))

Bad IDs: 0
Rows left: 5000


## Step 5 - Fix text columns (casing, typos)

In [10]:
# Title Case for simple text columns: "MAMMAL", " mammal " -> "Mammal"
for col in ["common_name", "class", "diet", "habitat", "facility", "health_status"]:
    df[col] = df[col].str.title()

# scientific name: "panthera leo" -> "Panthera leo"
df["scientific_name"] = df["scientific_name"].str.capitalize()

df["class"].value_counts()

,count
class,
Mammal,2834
Reptile,857
Bird,796
Fish,323
Amphibian,190


### Fix typos using the scientific name
`scientific_name` is reliable, and every species has one common name, class, diet, habitat and status.
So we take the **most common value (mode)** for each species and use it to fix typos like `Lyon` -> `Lion` and to fill gaps.

In [11]:
def most_common(series):
    modes = series.dropna().mode()
    return modes.iloc[0] if len(modes) > 0 else np.nan

for col in ["common_name", "class", "diet", "habitat"]:
    df[col] = df.groupby("scientific_name")[col].transform(most_common)

df["common_name"].nunique()   # should be 40 species

41

## Step 6 - Standardize categories with a dictionary

In [12]:
# Country: many spellings -> one spelling
country_map = {
    "usa": "USA", "us": "USA", "u.s.a.": "USA", "united states": "USA",
    "ind": "India", "s. africa": "South Africa",
    "drc": "Congo", "dr congo": "Congo",
    "russian federation": "Russia",
}
country = df["country"].str.lower()
df["country"] = country.map(country_map).fillna(df["country"].str.title())

# Conservation status: codes and names -> full names
status_map = {
    "lc": "Least Concern", "least concern": "Least Concern",
    "nt": "Near Threatened", "near threatened": "Near Threatened",
    "vu": "Vulnerable", "vulnerable": "Vulnerable",
    "en": "Endangered", "endangered": "Endangered",
    "cr": "Critically Endangered", "critically endangered": "Critically Endangered",
}
df["conservation_status"] = df["conservation_status"].str.lower().map(status_map)
df["conservation_status"] = df.groupby("scientific_name")["conservation_status"].transform(most_common)

# Sex: M / male / FEMALE ... -> Male / Female (look at the first letter)
df["sex"] = df["sex"].str.lower().str[0].map({"m": "Male", "f": "Female"}).fillna("Unknown")

# Vaccinated: Yes / Y / 1 / True -> True
vacc_map = {"yes": True, "y": True, "1": True, "true": True,
            "no": False, "n": False, "0": False, "false": False}
df["vaccinated"] = df["vaccinated"].str.lower().map(vacc_map)

df[["country", "conservation_status", "sex", "vaccinated"]].nunique()

,0
country,29
conservation_status,5
sex,3
vaccinated,2


## Step 6b - Check the categories
Always print `value_counts()` to confirm the fix worked.

In [13]:
print(df["country"].value_counts().head(10))
print(df["sex"].value_counts())

country
Australia    534
Kenya        446
USA          358
Indonesia    349
Tanzania     311
Mexico       300
India        248
Pakistan     232
Canada       217
Russia       203
Name: count, dtype: int64
sex
Male       2426
Female     2424
Unknown     150
Name: count, dtype: int64


## Step 7 - Fix number columns
Numbers are stored as text like `"120 kg"`, `"294 lbs"`, `"$216.29"`. We write small functions to turn them into real numbers.

In [14]:
def to_number(value):
    """'$1,216.29' -> 1216.29 ,  '15 cm' -> 15.0 ,  'abc' -> NaN"""
    if pd.isna(value):
        return np.nan
    text = str(value).replace(",", "")
    found = re.findall(r"-?\d+\.?\d*", text)
    return float(found[0]) if found else np.nan

def to_kg(value):
    """Convert '294 lbs', '7 g', '120 kg', '120' -> kg"""
    if pd.isna(value):
        return np.nan
    text = str(value).lower()
    number = to_number(text)
    if "lbs" in text:
        number = number * 0.4536
    elif text.endswith("g") and not text.endswith("kg"):
        number = number / 1000
    return number

def age_to_number(value):
    """'5.5 yrs' -> 5.5, 'three' -> 3"""
    words = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
    if pd.isna(value):
        return np.nan
    text = str(value).lower()
    return words[text] if text in words else to_number(text)

df["weight_kg"] = df["weight_kg"].apply(to_kg)
df["age_years"] = df["age_years"].apply(age_to_number)

for col in ["body_length_cm", "daily_food_kg", "checkup_cost_usd", "lifespan_years"]:
    df[col] = df[col].apply(to_number)

df.describe().T

,count,mean,std,min,25%,50%,75%,max
age_years,4607.0,15.535924,32.030826,-2.00,5.5000,10.300,19.20000,999.00
weight_kg,4700.0,718.168808,3692.572238,-120.50,16.4925,79.690,368.09178,151025.00
body_length_cm,4750.0,240.145368,292.865223,-10.00,108.6000,160.750,319.47500,9999.00
lifespan_years,5000.0,33.777600,35.764541,-1.00,15.0000,25.000,40.00000,500.00
daily_food_kg,4508.0,31.651764,77.854878,0.00,0.7400,4.130,16.32000,680.52
checkup_cost_usd,4600.0,238.383672,270.493391,15.36,86.9225,151.285,271.53750,3120.99


## Step 8 - Remove impossible values
Look at the `min` and `max` above: negative weights, `99999`, ages of `999`.
These are errors, so we set them to `NaN`.
Rule: a value must be **above 0** and **not more than 3x its species' median**.

In [15]:
# Lifespan is fixed per species, so use the most common value
df.loc[df["lifespan_years"] <= 0, "lifespan_years"] = np.nan
df["lifespan_years"] = df.groupby("scientific_name")["lifespan_years"].transform(most_common)

# Weight, length, food, cost
for col in ["weight_kg", "body_length_cm", "daily_food_kg", "checkup_cost_usd"]:
    typical = df[col].where(df[col] > 0).groupby(df["scientific_name"]).transform("median")
    too_big = df[col] > 3 * typical
    too_small = df[col] <= 0
    df.loc[too_big | too_small, col] = np.nan

# An animal cannot be older than its species lifespan
df.loc[(df["age_years"] < 0) | (df["age_years"] > df["lifespan_years"]), "age_years"] = np.nan

df.describe().T

,count,mean,std,min,25%,50%,75%,max
age_years,4594.0,14.614062,13.172380,0.10,5.500,10.300,19.1000,88.20
weight_kg,4670.0,618.811691,1567.195410,0.01,19.280,80.230,367.6995,11600.73
body_length_cm,4740.0,234.325422,189.972022,4.90,108.875,160.800,318.9500,1187.30
lifespan_years,5000.0,32.355200,22.199162,5.00,15.000,25.000,40.0000,100.00
daily_food_kg,4336.0,32.907322,79.123658,0.01,1.160,4.775,16.9600,680.52
checkup_cost_usd,4566.0,234.515644,260.008059,15.36,86.550,150.215,266.5475,2922.49


## Step 9 - Fix dates
The dates come in 5 different formats. We try each format in turn.

In [16]:
formats = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d %b %Y", "%B %d, %Y"]

clean_date = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for fmt in formats:
    attempt = pd.to_datetime(df["date_recorded"], format=fmt, errors="coerce")
    clean_date = clean_date.fillna(attempt.astype("datetime64[ns]"))

# dates in the future are errors
clean_date = clean_date.where(clean_date <= pd.Timestamp.today())

df["date_recorded"] = clean_date
df["date_recorded"].describe()

,date_recorded
count,4892
mean,2023-11-16 07:34:11.676206080
min,2022-01-01 00:00:00
25%,2022-12-19 00:00:00
50%,2023-11-21 12:00:00
75%,2024-10-21 00:00:00
max,2025-09-11 00:00:00


## Step 10 - Fill the remaining missing values
Filling = guessing, so keep it simple and do it on purpose:
numbers get the **species median**, and text columns get `"Unknown"`.

In [17]:
num_cols = ["age_years", "weight_kg", "body_length_cm", "daily_food_kg", "checkup_cost_usd"]
for col in num_cols:
    species_median = df.groupby("scientific_name")[col].transform("median")
    df[col] = df[col].fillna(species_median)

for col in ["country", "facility", "health_status"]:
    df[col] = df[col].fillna("Unknown")

df.isna().sum()

,0
animal_id,0
common_name,0
scientific_name,0
class,0
diet,0
habitat,0
country,0
facility,0
conservation_status,0
sex,0


## Step 11 - Final check and save

In [18]:
df = df.reset_index(drop=True)
print("Final shape:", df.shape)
print("Duplicate IDs:", df["animal_id"].duplicated().sum())
df.info()

Final shape: (5000, 19)
Duplicate IDs: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   animal_id            5000 non-null   object        
 1   common_name          5000 non-null   object        
 2   scientific_name      5000 non-null   object        
 3   class                5000 non-null   object        
 4   diet                 5000 non-null   object        
 5   habitat              5000 non-null   object        
 6   country              5000 non-null   object        
 7   facility             5000 non-null   object        
 8   conservation_status  5000 non-null   object        
 9   sex                  5000 non-null   object        
 10  age_years            5000 non-null   float64       
 11  weight_kg            5000 non-null   float64       
 12  body_length_cm       5000 non-null   float64     

In [19]:
df.to_csv("animals_clean.csv", index=False)
from google.colab import files
files.download("animals_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
dfs = pd.read_csv("/content/animals_clean.csv").head()
dfs


,animal_id,common_name,scientific_name,class,diet,habitat,country,facility,conservation_status,sex,age_years,weight_kg,body_length_cm,lifespan_years,daily_food_kg,health_status,vaccinated,checkup_cost_usd,date_recorded
0,ZA-04453,Tiger,Panthera tigris,Mammal,Carnivore,Forest,India,Metro City Zoo,Endangered,Male,10.90,297.52000,291.5,16.0,14.240,Healthy,True,216.29,2025-06-21
1,ZA-00434,Giant Panda,Ailuropoda melanoleuca,Mammal,Herbivore,Forest,China,Greenfield Sanctuary,Vulnerable,Male,10.10,105.82000,165.4,20.0,5.825,Healthy,True,129.65,2025-08-11
2,ZA-02153,Lion,Panthera leo,Mammal,Carnivore,Savanna,Tanzania,Greenfield Sanctuary,Vulnerable,Female,5.80,180.22000,230.3,14.0,5.720,Healthy,False,211.39,2022-05-16
3,ZA-04341,Plains Zebra,Equus quagga,Mammal,Herbivore,Savanna,Tanzania,Coastal Aquarium & Zoo,Near Threatened,Female,9.75,133.72128,176.0,25.0,7.830,Healthy,True,137.20,2023-09-03
4,ZA-02905,Chimpanzee,Pan troglodytes,Mammal,Omnivore,Forest,Uganda,Metro City Zoo,Endangered,Female,20.90,48.55616,86.1,40.0,2.220,Healthy,True,87.31,2023-01-21
